# Phase 1: Production LLM — Inference Optimisation

This notebook covers how LLMs are served efficiently in production. Unlike Sections 8–12 which focused on training and prompting, this section is about **what happens after you have a model** and need to serve it to users at scale without burning your compute budget.

These topics come up in AI Engineer interviews in system design rounds: *"How would you serve this model to 10,000 concurrent users?"* or *"Our inference costs are too high — what would you do?"*

No model loading or GPU required — demos use the Anthropic API or pure Python/NumPy to illustrate concepts.

---

## Contents
- **Section 13:** KV Cache
- **Section 14:** Quantisation at Inference (INT8 / INT4)
- **Section 15:** Batching Strategies
- **Section 16:** Speculative Decoding
- **Section 17:** Prompt Caching

---

# Section 13: KV Cache

## Theory

### The Problem: Redundant Computation

Recall how autoregressive generation works: the model generates one token at a time, and each new token requires a full forward pass over the entire sequence so far. If your prompt is 500 tokens and you're generating 200 tokens of output, you run 200 forward passes. In each pass, every token attends to every previous token.

The attention computation for a single layer is:
```
Attention(Q, K, V) = softmax(QK^T / √d_k) * V
```

The K (key) and V (value) matrices for the prompt tokens are **identical on every generation step**. Token 1's key and value vectors don't change between step 1 and step 200 — the prompt is fixed. Without caching, you're recomputing them 200 times. This is pure wasted compute.

### The Solution: Cache the Keys and Values

The KV cache stores the K and V tensors for every token that has already been processed. On each generation step, you only compute Q, K, V for the **new token**, look up the cached K and V for all previous tokens, concatenate them, and run attention. This reduces the attention computation from O(n²) per step to O(n) per step, where n is the sequence length so far.

The forward pass structure changes from:
```
Step 1: compute K,V for tokens [1..500]        → store in cache
Step 2: compute K,V for tokens [1..501]        → recomputing 1..500 wastefully
Step 3: compute K,V for tokens [1..502]        → recomputing 1..501 wastefully
```

To:
```
Step 1: compute K,V for tokens [1..500]        → store in cache
Step 2: compute K,V for token [501] only       → look up 1..500 from cache
Step 3: compute K,V for token [502] only       → look up 1..501 from cache
```

### Memory Cost

The KV cache isn't free — it trades compute for memory. The cache size for a single sequence is:

```
cache_size = 2 × num_layers × num_heads × head_dim × seq_len × bytes_per_element
```

The factor of 2 is for K and V (we don't cache Q — queries are only needed for the current token). For a 7B parameter model with 32 layers, 32 heads, head_dim=128, seq_len=4096, in fp16:

```
2 × 32 × 32 × 128 × 4096 × 2 bytes = ~2GB per sequence
```

This is why long context is expensive. A 128K context window with the same model would require ~64GB just for the KV cache of a single sequence. In production you need to manage cache memory carefully — this is why serving systems have maximum batch sizes and context length limits.

### Prefill vs Decode

Modern inference systems split generation into two phases:

- **Prefill phase**: process the entire prompt in one forward pass (parallelisable across all prompt tokens), populate the KV cache. This is compute-bound.
- **Decode phase**: generate one token at a time using the cache. This is memory-bandwidth-bound — you're reading the entire KV cache from GPU memory on every step.

This distinction matters for system design: prefill and decode have different bottlenecks and can be optimised differently. Some production systems even run them on separate hardware.

### Why This Matters for AI Engineer Interviews

If asked "why is generating long outputs slow?", the answer is memory bandwidth, not compute. The KV cache grows with sequence length and reading it from GPU memory on every decode step becomes the bottleneck. Techniques like **multi-query attention** (MQA) and **grouped-query attention** (GQA) reduce KV cache size by sharing K and V heads across query heads — this is why Llama 2 and Mistral use GQA.

### KV Cache — Clarifications

### Prompt Processing
The entire prompt is processed in a **single forward pass** during prefill — all tokens
simultaneously, just like training. The causal mask enforces the autoregressive constraint
(token N only attends to tokens 1..N) but computationally it all happens in parallel.
This is what makes prefill fast.

### Two Things Called "K and V"
There is an important distinction between:

**Weight matrices** — `W_K` and `W_V`. Learned model parameters, shape `(d_model, d_k)`.
Never change after training. Every forward pass uses the same `W_K` and `W_V`.
These are NOT what gets cached.

**Computed K,V vectors** — the result of multiplying each token's hidden state through
those weight matrices:
```python
K_token1 = hidden_state_token1 @ W_K    # shape: (1, d_k)
V_token1 = hidden_state_token1 @ W_V    # shape: (1, d_v)
```
These are token-specific and ARE what gets cached.

### Why Computed K,V Vectors Don't Change
Token 1's hidden state is determined by its embedding and the attention computations in
layers below. Since token 1's input never changes (it's always the same prompt token),
and the weight matrices never change, `K_token1` and `V_token1` are always identical on
every decode step. Recomputing them is pure waste — hence the cache.

### Shapes
The weight matrices per head:

W_Q: (d_model, d_k)

W_K: (d_model, d_k)

W_V: (d_model, d_v)

Where d_k = d_v = d_model / num_heads.

During prefill, computed across the full sequence:
hidden_states:        (batch, seq_len, d_model)

Q = hidden_states @ W_Q  →  (batch, seq_len, d_k)

K = hidden_states @ W_K  →  (batch, seq_len, d_k)

V = hidden_states @ W_V  →  (batch, seq_len, d_v)
Reshaped for multi-head attention:

Q, K, V  →  (batch, num_heads, seq_len, head_dim)

### What the KV Cache Stores
Per layer, per head, the cache grows as tokens are generated:

K cache: (batch, num_heads, seq_len_so_far, head_dim) # conceptually this makes sense to me matmul by Q(batch,num_heads,1,head_dim) because its one Q being compared to against all previous tokens keys so a vector * matrix

V cache: (batch, num_heads, seq_len_so_far, head_dim)

During decode step t, only the new token's K and V are computed:

new_K: (batch, num_heads, 1, head_dim)  →  appended to K cache

new_V: (batch, num_heads, 1, head_dim)  →  appended to V cache

Attention for the new token:

Q:          (batch, num_heads, 1, head_dim)

Q @ K.T  →  (batch, num_heads, 1, seq_len_so_far+1)   # attention scores

scores @ V → (batch, num_heads, 1, head_dim)           # attended output

In practice frameworks pre-allocate the cache at shape
`(batch, num_heads, max_len, head_dim)` and fill it in as generation proceeds.
Only `seq_len_so_far` positions are populated at any given step — the rest is
reserved but empty. This pre-allocation waste is exactly what PagedAttention fixes.

### What Happens to the Attention Output
The output `(batch, num_heads, 1, head_dim)` is reshaped back to `(batch, 1, d_model)`
— the new token's updated hidden state. It flows through the rest of that layer
(feed-forward, layer norm) and up to the next transformer layer where the whole process
repeats with that layer's own W_Q, W_K, W_V.

After all N layers, the final hidden state passes through the unembedding layer:
(batch, 1, d_model) @ W_unembed  →  (batch, 1, vocab_size)

Softmax → sample → next token ID. The hidden states themselves do not accumulate.
Only the KV cache does — appended at the start of the next decode step.

### Why Q is Not Cached
Q is the search query for the **current token** — what it's looking for right now.
It's use-once. Token 502 has its own Q and doesn't care what token 501 was looking for.
Crucially, Q never attends to other Q vectors. The attention formula is:

softmax(Q @ K.T) @ V

Q only ever interacts with K and V. There is no Q-to-Q operation anywhere in the
attention mechanism.

### Output Per Decode Step
Each decode step outputs **one token per sequence**:

(batch, 1, vocab_size)  →  one token ID per sequence in the batch

With batch=8, you get 8 tokens out per step — one per sequence, all in parallel.
This is why batching increases throughput: the GPU does roughly the same work per
step whether batch=1 or batch=8, so you generate 8 tokens for the cost of ~1 step.

### Which Layers?
All of them. Every transformer layer has its own W_K, W_V, W_Q and therefore its own
set of computed K,V vectors per token. This is why the memory formula contains
`num_layers`:

cache_size = 2 × num_layers × num_heads × head_dim × seq_len × bytes_per_element

The `2` is for K and V. `num_layers` is because each layer caches independently.
A 32-layer model has 32 separate K caches and 32 separate V caches, all growing
as the sequence gets longer.

In [1]:
import numpy as np
import time

# Simulate the compute savings from KV caching
# We'll model the number of K,V computations with and without caching

def compute_without_kv_cache(prompt_len: int, gen_len: int) -> int:
    """
    Without KV cache: at each generation step t, we recompute K,V
    for all tokens 1..(prompt_len + t).
    Total K,V computations = sum of sequence lengths across all steps.
    """
    total = 0
    for step in range(gen_len):
        seq_len = prompt_len + step
        total += seq_len  # recompute K,V for all tokens
    return total

def compute_with_kv_cache(prompt_len: int, gen_len: int) -> int:
    """
    With KV cache: prefill computes K,V for all prompt tokens once.
    Each decode step only computes K,V for the single new token.
    """
    prefill = prompt_len        # compute K,V for all prompt tokens once
    decode  = gen_len * 1      # compute K,V for 1 new token per step
    return prefill + decode

# Compare across different sequence lengths
print(f"{'Prompt':>8} {'Gen':>6} {'No Cache':>12} {'With Cache':>12} {'Speedup':>10}")
print("-" * 52)

configs = [
    (100,  50),
    (500,  200),
    (1000, 500),
    (4000, 1000),
    (8000, 2000),
]

for prompt_len, gen_len in configs:
    no_cache   = compute_without_kv_cache(prompt_len, gen_len)
    with_cache = compute_with_kv_cache(prompt_len, gen_len)
    speedup    = no_cache / with_cache
    print(f"{prompt_len:>8} {gen_len:>6} {no_cache:>12,} {with_cache:>12,} {speedup:>9.1f}x")

  Prompt    Gen     No Cache   With Cache    Speedup
----------------------------------------------------
     100     50        6,225          150      41.5x
     500    200      119,900          700     171.3x
    1000    500      624,750        1,500     416.5x
    4000   1000    4,499,500        5,000     899.9x
    8000   2000   17,999,000       10,000    1799.9x


In [2]:
# Model the KV cache memory footprint for different model sizes
# Formula: 2 × num_layers × num_heads × head_dim × seq_len × bytes_per_element

def kv_cache_memory_gb(
    num_layers: int,
    num_heads: int,
    head_dim: int,
    seq_len: int,
    bytes_per_element: int = 2  # fp16
) -> float:
    total_bytes = 2 * num_layers * num_heads * head_dim * seq_len * bytes_per_element
    return total_bytes / (1024 ** 3)

# Approximate configs for common models
models = [
    {"name": "GPT-2 (117M)",    "layers": 12,  "heads": 12,  "head_dim": 64},
    {"name": "Llama-2 7B",      "layers": 32,  "heads": 32,  "head_dim": 128},
    {"name": "Llama-2 13B",     "layers": 40,  "heads": 40,  "head_dim": 128},
    {"name": "Llama-2 70B",     "layers": 80,  "heads": 64,  "head_dim": 128},
]

seq_lengths = [2048, 8192, 32768, 131072]

print(f"KV Cache Memory (GB) per sequence — fp16")
print(f"{'Model':<22}" + "".join(f"{s:>10,}" for s in seq_lengths))
print("-" * 62)

for m in models:
    row = f"{m['name']:<22}"
    for s in seq_lengths:
        gb = kv_cache_memory_gb(m["layers"], m["heads"], m["head_dim"], s)
        row += f"{gb:>9.2f}G"
    print(row)

print()
print("A single A100 80GB GPU has 80GB VRAM total.")
print("Model weights alone take most of that — KV cache competes directly with batch size.")

KV Cache Memory (GB) per sequence — fp16
Model                      2,048     8,192    32,768   131,072
--------------------------------------------------------------
GPT-2 (117M)               0.07G     0.28G     1.12G     4.50G
Llama-2 7B                 1.00G     4.00G    16.00G    64.00G
Llama-2 13B                1.56G     6.25G    25.00G   100.00G
Llama-2 70B                5.00G    20.00G    80.00G   320.00G

A single A100 80GB GPU has 80GB VRAM total.
Model weights alone take most of that — KV cache competes directly with batch size.


## ✅ What You Just Learned — Section 13

**For AI Engineer interviews:**
- KV cache is the single most important inference optimisation to know. It comes up in almost every system design question about LLM serving
- The key insight: K and V vectors for previous tokens never change, so recomputing them is pure waste
- The tradeoff is memory: KV cache grows linearly with sequence length, which is why long-context serving is expensive
- Prefill (compute-bound) and decode (memory-bandwidth-bound) have different bottlenecks — mentioning this distinction signals depth
- GQA/MQA are architectural responses to KV cache memory pressure — worth knowing as follow-up

---

# Section 14: Quantisation at Inference (INT8 / INT4)

## Theory

### What Quantisation Is

Model weights are stored as floating point numbers. By default, most models use fp32 (32-bit) or fp16/bf16 (16-bit). Quantisation compresses these weights to lower bit widths — INT8 (8-bit integers) or INT4 (4-bit integers) — reducing memory usage and often increasing throughput.

Think of it like compressing an image: you lose some precision, but the file is much smaller and loads faster. The question is how much precision loss you can tolerate before model quality degrades noticeably.

### The Math: How Floating Point Becomes Integer

A floating point weight `w` is mapped to an integer `q` via:
```
q = round(w / scale) + zero_point
w ≈ (q - zero_point) × scale
```

Where `scale` and `zero_point` are calibration parameters computed from the distribution of weights in each layer. During inference, you dequantise back to fp16/fp32 for the actual matrix multiply, or use integer arithmetic directly if hardware supports it.

### Inference Quantisation vs QLoRA Quantisation

You've already seen quantisation in Section 11 (QLoRA). It's worth distinguishing the two uses:

- **QLoRA quantisation**: applied during *training*, compresses the frozen base model weights to fit in memory so you can fine-tune on a consumer GPU. The LoRA adapters remain in higher precision.
- **Inference quantisation**: applied at *serving time*, compresses the deployed model to reduce memory and increase throughput. No training involved.

Same technique, different context and goals.

### INT8 vs INT4

**INT8:**
- 2× memory reduction vs fp16
- Minimal quality degradation on most tasks
- Well supported by modern hardware (NVIDIA A100, H100 have INT8 tensor cores)
- Safe default for production

**INT4:**
- 4× memory reduction vs fp16
- More noticeable quality degradation, especially on reasoning tasks
- Requires more careful calibration (GPTQ, AWQ are popular methods)
- Used when you absolutely need to fit a larger model into available VRAM

### Quantisation Methods Worth Knowing

- **RTN (Round-to-Nearest)**: simplest approach, round each weight directly. Fast but degrades quality at INT4
- **GPTQ**: uses second-order gradient information to minimise quantisation error per layer. Better quality at INT4 than RTN
- **AWQ (Activation-aware Weight Quantisation)**: identifies which weights are most important based on activation magnitudes and protects them. State of the art for INT4
- **bitsandbytes**: the library we used in Section 11, implements LLM.int8() which handles outlier activations that naive INT8 quantisation struggles with

### Activation Quantisation vs Weight-Only Quantisation

**Weight-only quantisation**: only compress the stored weights. Dequantise to fp16 before matrix multiply. Reduces memory and load time, helps with memory-bandwidth bottleneck in decode phase.

**Activation quantisation**: quantise both weights and activations. Enables true INT8 matrix multiply on hardware that supports it (NVIDIA tensor cores). Higher throughput but harder to implement without quality loss because activations have dynamic ranges that are harder to calibrate than weights.

### The Outlier Problem

LLMs have a specific challenge: a small number of activation dimensions have very large magnitudes (outliers) while most are small. Naive INT8 quantisation picks a scale factor based on the max value — the outliers dominate the scale, and all the small values get rounded to zero, destroying model quality. LLM.int8() solves this by keeping outlier dimensions in fp16 and quantising the rest to INT8 (mixed-precision decomposition).

---

For a product-focused AI Engineer role you need to know three things about quantisation:
1. What it is in one sentence-
Compressing model weights from floating point (fp16/fp32) to lower bit integers (INT8/INT4) to reduce memory usage.

2. The tradeoff 
- INT8 — 2x memory reduction, minimal quality loss, safe default.
- INT4 — 4x memory reduction, more quality loss, use when you need to fit a larger model.
3. The connection to your Section 11 work - bitsandbytes is what enables QLoRA — it quantises the frozen base model to 4-bit so it fits in memory during fine-tuning. You've already used it, now you know why.

In [3]:
import numpy as np

# Simulate the quantisation and dequantisation process
# This shows exactly what happens to weights when you quantise to INT8

def quantise_int8(weights: np.ndarray):
    """
    Symmetric INT8 quantisation.
    Maps weights to [-127, 127] range.
    Returns quantised weights and scale factor.
    """
    max_val = np.max(np.abs(weights))
    scale   = max_val / 127.0
    q       = np.round(weights / scale).astype(np.int8)
    return q, scale

def dequantise_int8(q: np.ndarray, scale: float) -> np.ndarray:
    """Reconstruct approximate fp32 weights from INT8."""
    return q.astype(np.float32) * scale

def quantise_int4(weights: np.ndarray):
    """
    Symmetric INT4 quantisation.
    Maps weights to [-7, 7] range (4-bit signed = -8 to 7, use -7 to 7 for symmetry).
    """
    max_val = np.max(np.abs(weights))
    scale   = max_val / 7.0
    q       = np.clip(np.round(weights / scale), -7, 7).astype(np.int8)
    return q, scale

# Simulate a small weight matrix (e.g. a slice of an attention projection)
np.random.seed(42)
weights_fp32 = np.random.randn(8, 8).astype(np.float32) * 0.1

# Quantise
q_int8, scale_int8 = quantise_int8(weights_fp32)
q_int4, scale_int4 = quantise_int4(weights_fp32)

# Dequantise
weights_int8_reconstructed = dequantise_int8(q_int8, scale_int8)
weights_int4_reconstructed = dequantise_int8(q_int4, scale_int4)

# Measure error
err_int8 = np.mean(np.abs(weights_fp32 - weights_int8_reconstructed))
err_int4 = np.mean(np.abs(weights_fp32 - weights_int4_reconstructed))

print("Original weights (fp32):")
print(weights_fp32.round(4))
print(f"\nINT8 quantised then reconstructed:")
print(weights_int8_reconstructed.round(4))
print(f"Mean absolute error (INT8): {err_int8:.6f}")
print(f"\nINT4 quantised then reconstructed:")
print(weights_int4_reconstructed.round(4))
print(f"Mean absolute error (INT4): {err_int4:.6f}")

Original weights (fp32):
[[ 0.0497 -0.0138  0.0648  0.1523 -0.0234 -0.0234  0.1579  0.0767]
 [-0.0469  0.0543 -0.0463 -0.0466  0.0242 -0.1913 -0.1725 -0.0562]
 [-0.1013  0.0314 -0.0908 -0.1412  0.1466 -0.0226  0.0068 -0.1425]
 [-0.0544  0.0111 -0.1151  0.0376 -0.0601 -0.0292 -0.0602  0.1852]
 [-0.0013 -0.1058  0.0823 -0.1221  0.0209 -0.196  -0.1328  0.0197]
 [ 0.0738  0.0171 -0.0116 -0.0301 -0.1479 -0.072  -0.0461  0.1057]
 [ 0.0344 -0.1763  0.0324 -0.0385 -0.0677  0.0612  0.1031  0.0931]
 [-0.0839 -0.0309  0.0331  0.0976 -0.0479 -0.0186 -0.1106 -0.1196]]

INT8 quantised then reconstructed:
[[ 0.0494 -0.0139  0.0648  0.1528 -0.0231 -0.0231  0.1574  0.0772]
 [-0.0463  0.054  -0.0463 -0.0463  0.0247 -0.1913 -0.1728 -0.0555]
 [-0.1018  0.0309 -0.091  -0.142   0.1466 -0.0231  0.0062 -0.142 ]
 [-0.054   0.0108 -0.1157  0.037  -0.0602 -0.0293 -0.0602  0.1852]
 [-0.0015 -0.1065  0.0818 -0.1219  0.0216 -0.196  -0.1327  0.0201]
 [ 0.0741  0.017  -0.0108 -0.0309 -0.1481 -0.0725 -0.0463  0.1065]


In [4]:
# Memory savings from quantisation across model sizes

def model_memory_gb(params: int, bits: int) -> float:
    """Approximate VRAM needed to load a model at a given bit width."""
    bytes_per_param = bits / 8
    return (params * bytes_per_param) / (1024 ** 3)

models = [
    ("7B",   7_000_000_000),
    ("13B", 13_000_000_000),
    ("70B", 70_000_000_000),
]

print(f"{'Model':<8} {'fp32 (32b)':>12} {'fp16 (16b)':>12} {'INT8 (8b)':>12} {'INT4 (4b)':>12}")
print("-" * 58)

for name, params in models:
    fp32 = model_memory_gb(params, 32)
    fp16 = model_memory_gb(params, 16)
    int8 = model_memory_gb(params, 8)
    int4 = model_memory_gb(params, 4)
    print(f"{name:<8} {fp32:>10.1f}GB {fp16:>10.1f}GB {int8:>10.1f}GB {int4:>10.1f}GB")

print()
print("A100 40GB VRAM limits:")
print("  fp16: can barely fit 13B, cannot fit 70B")
print("  INT8: can fit 13B comfortably, cannot fit 70B")
print("  INT4: can fit 70B (just barely) on a single A100 40GB")

Model      fp32 (32b)   fp16 (16b)    INT8 (8b)    INT4 (4b)
----------------------------------------------------------
7B             26.1GB       13.0GB        6.5GB        3.3GB
13B            48.4GB       24.2GB       12.1GB        6.1GB
70B           260.8GB      130.4GB       65.2GB       32.6GB

A100 40GB VRAM limits:
  fp16: can barely fit 13B, cannot fit 70B
  INT8: can fit 13B comfortably, cannot fit 70B
  INT4: can fit 70B (just barely) on a single A100 40GB


## ✅ What You Just Learned — Section 14

**For AI Engineer interviews:**
- Quantisation is the primary lever for fitting larger models into available VRAM
- INT8 is the safe default — minimal quality loss, good hardware support
- INT4 gets you 4× compression but requires careful calibration (GPTQ, AWQ)
- The outlier problem is a specific LLM challenge that naive quantisation fails on — LLM.int8() solves it via mixed-precision decomposition
- Distinguish inference quantisation (serving) from QLoRA quantisation (training) — same technique, different goals
- Being able to explain the scale/zero_point math shows you understand what's actually happening

---

# Section 15: Batching Strategies

## Theory

### Why Batching Matters

GPUs are designed for massive parallelism. Running one request at a time wastes most of that parallelism — the GPU is mostly idle waiting for the next token to generate. Batching multiple requests together lets the GPU do useful work on all of them simultaneously, dramatically increasing throughput (requests per second) at the cost of slightly higher latency per request.

The throughput/latency tradeoff is the central tension in LLM serving:
- **Small batch / no batch**: lowest latency, terrible throughput, high cost per request
- **Large batch**: highest throughput, higher latency, low cost per request

### Static Batching

The naive approach: collect N requests, pad them all to the same length (the longest sequence), process them together, return all results when the longest sequence finishes.

**Problem**: requests finish at different times. If one request in a batch of 8 needs 500 tokens and the others only need 50, the GPU sits idle for 7 requests while it finishes the long one. The padding tokens consume memory and compute but produce nothing useful. In practice, request lengths vary enormously, making static batching highly inefficient.

### Continuous Batching (Dynamic Batching)

The modern solution, pioneered by the Orca paper (2022). Instead of waiting for all requests in a batch to finish, the system processes requests at the **iteration level** — after each token generation step, finished sequences are removed from the batch and new requests are inserted immediately.

The batch composition changes dynamically on every step:
```
Step 1:  [req_A, req_B, req_C, req_D]       ← 4 active requests
Step 2:  [req_A, req_B, req_C, req_D]       ← all still generating
Step 3:  [req_A, req_B, ------, req_D]      ← req_C finished, slot free
Step 4:  [req_A, req_B, req_E,  req_D]      ← req_E immediately inserted
```

This eliminates the padding waste and keeps GPU utilisation high. vLLM, TGI (Text Generation Inference), and most modern serving frameworks implement continuous batching.

### PagedAttention

Continuous batching solved the compute efficiency problem. PagedAttention (introduced with vLLM) solved the memory efficiency problem.

With a standard KV cache, memory is allocated upfront for the maximum possible sequence length. Most sequences don't reach the maximum, so memory is wasted. Worse, fragmentation means you can't always fit as many sequences in memory as you theoretically should.

PagedAttention borrows the idea of virtual memory from operating systems. The KV cache is divided into fixed-size **pages** (like memory pages). Each sequence gets pages allocated on demand as it grows, rather than a large contiguous block upfront. Pages from different sequences can be interleaved in physical memory — no contiguous allocation required.

This allows:
- Near-zero memory waste from pre-allocation
- **KV cache sharing** between requests with the same prefix (e.g. a system prompt shared across all users)
- Higher effective batch sizes for the same VRAM

### Chunked Prefill

A more recent optimisation: split long prompts into chunks and interleave prefill and decode steps across the batch. This prevents a single long prompt from monopolising the GPU during its prefill phase, reducing latency spikes for other requests in the batch.

---

In [ ]:
import numpy as np

# Simulate static vs continuous batching throughput
# We model GPU utilisation and time-to-completion for a stream of requests

np.random.seed(42)

# Simulate 20 requests with varying output lengths
num_requests = 20
output_lengths = np.random.randint(20, 300, size=num_requests)

print("Simulated request output lengths (tokens):")
print(output_lengths)
print(f"Min: {output_lengths.min()}, Max: {output_lengths.max()}, "
      f"Mean: {output_lengths.mean():.0f}")

# Static batching: batch of 4, wait for all to finish before next batch
batch_size = 4
static_total_steps = 0
static_wasted_steps = 0

for i in range(0, num_requests, batch_size):
    batch = output_lengths[i:i+batch_size]
    max_len = batch.max()           # GPU runs until longest finishes
    static_total_steps  += max_len * len(batch)   # total compute slots used
    static_wasted_steps += sum(max_len - l for l in batch)  # padding waste

static_utilisation = 1 - (static_wasted_steps / static_total_steps)

# Continuous batching: slots freed immediately, new requests inserted
# Model: always keep batch_size active requests, never idle
# Total steps = sum of all output lengths (no wasted slots)
continuous_useful_steps = output_lengths.sum()
# With continuous batching, GPU is always busy (assuming enough requests)
continuous_utilisation = 1.0  # ideal case

print(f"\n--- Static Batching (batch_size={batch_size}) ---")
print(f"Total compute slots used: {static_total_steps:,}")
print(f"Wasted (padding) slots:   {static_wasted_steps:,}")
print(f"GPU utilisation:          {static_utilisation:.1%}")

print(f"\n--- Continuous Batching ---")
print(f"Total useful compute:     {continuous_useful_steps:,}")
print(f"Wasted slots:             0")
print(f"GPU utilisation:          {continuous_utilisation:.1%} (ideal)")

print(f"\nContinuous batching reduces wasted compute by: {static_wasted_steps:,} steps")
print(f"Throughput improvement potential: {1/static_utilisation:.2f}x")

In [ ]:
# Simulate PagedAttention memory savings vs static KV cache allocation

# Parameters
max_seq_len     = 2048   # maximum context length
actual_lengths  = np.random.randint(100, 800, size=16)  # actual request lengths
bytes_per_token = 1024   # approximate KV cache bytes per token (simplified)

# Static allocation: each request gets max_seq_len slots upfront
static_memory_bytes  = len(actual_lengths) * max_seq_len * bytes_per_token
static_used_bytes    = actual_lengths.sum() * bytes_per_token
static_wasted_bytes  = static_memory_bytes - static_used_bytes

# Paged allocation: only allocate pages actually needed
page_size        = 16   # tokens per page
pages_needed     = np.ceil(actual_lengths / page_size).astype(int)
paged_memory     = pages_needed.sum() * page_size * bytes_per_token
paged_wasted     = paged_memory - static_used_bytes  # small internal fragmentation only

print("Request actual lengths:", actual_lengths)
print(f"\n--- Static KV Cache Allocation ---")
print(f"Allocated: {static_memory_bytes / 1024**2:.1f} MB")
print(f"Actually used: {static_used_bytes / 1024**2:.1f} MB")
print(f"Wasted: {static_wasted_bytes / 1024**2:.1f} MB "
      f"({100*static_wasted_bytes/static_memory_bytes:.0f}%)")

print(f"\n--- PagedAttention ---")
print(f"Allocated: {paged_memory / 1024**2:.1f} MB")
print(f"Actually used: {static_used_bytes / 1024**2:.1f} MB")
print(f"Internal fragmentation: {paged_wasted / 1024**2:.1f} MB "
      f"({100*paged_wasted/paged_memory:.1f}%)")
print(f"\nMemory saved vs static: {(static_memory_bytes - paged_memory) / 1024**2:.1f} MB")
print(f"Can fit {static_memory_bytes // paged_memory:.1f}x more requests in same VRAM")

## ✅ What You Just Learned — Section 15

**For AI Engineer interviews:**
- Static batching is simple but wasteful — padding waste and GPU idle time kill efficiency
- Continuous batching (Orca) is the modern standard — iteration-level scheduling, no padding waste
- PagedAttention (vLLM) solves the memory fragmentation problem — virtual memory for KV cache
- The combination of continuous batching + PagedAttention is why vLLM achieves 10-20x higher throughput than naive serving
- If asked "how would you serve an LLM at scale" — vLLM or TGI with continuous batching and paged attention is the answer

---

# Section 16: Speculative Decoding

## Theory

### The Problem: Decoding is Sequential

Autoregressive generation is fundamentally sequential — you cannot generate token N+1 until you have token N. This means no matter how fast your hardware is, you're bottlenecked by one forward pass per token. For a 70B model, each forward pass is slow, and generating 500 tokens means 500 serial forward passes.

Speculative decoding breaks this sequential bottleneck.

### The Key Insight

Most tokens in a sequence are easy to predict — common words, continuations of obvious phrases. You don't need a 70B model to predict "The" after "Once upon a time" or "is" after "Paris". A small, fast model can correctly guess many tokens that the large model would have generated anyway.

Speculative decoding exploits this:

1. A small **draft model** (e.g. 7B) generates K tokens speculatively in parallel — fast, cheap
2. The large **target model** (e.g. 70B) verifies all K tokens in a **single forward pass** — because verification is parallelisable across positions
3. Tokens that match the target model's distribution are accepted. The first rejected token is resampled from the target model's distribution
4. Repeat

In the best case (all K tokens accepted), you've generated K tokens with roughly the cost of one target model forward pass instead of K forward passes. Typical speedup is 2–3x on common tasks.

### Why Verification is Parallel

This is the clever part. During training, transformer forward passes process all tokens in parallel (that's what the causal mask enables). During inference, you can feed the entire draft sequence to the target model and get logits for all positions simultaneously — the same way training works. You then compare the target model's probability at each position to the draft model's choice.

The acceptance criterion uses a technique from rejection sampling:
```
Accept token t if: uniform_random(0,1) < p_target(t) / p_draft(t)
```

This guarantees that the final output distribution matches the target model exactly — speculative decoding doesn't change model quality, only speed.

### Draft Model Options

- **Separate smaller model**: e.g. use Llama-7B as draft for Llama-70B. Requires loading two models
- **Same model with early exit**: stop processing at an intermediate layer for the draft
- **Self-speculative / Medusa**: add draft heads directly to the target model that predict multiple future tokens simultaneously
- **Prompt lookup decoding**: for tasks where the output likely repeats from the input (summarisation, code completion), scan the prompt for matching n-grams to use as draft tokens — no draft model needed

### When It Helps (and Doesn't)

Speculative decoding works best when:
- The task has predictable, low-entropy outputs (code completion, translation, summarisation)
- The draft model is small enough that its forward pass is much faster than the target model
- You're in a latency-sensitive single-request setting (the speedup diminishes under heavy batching since the target model is already GPU-saturated)

It helps less when:
- Outputs are highly creative / high-entropy (the draft model guesses wrong more often)
- You're already running large batches (GPU is already busy; speculative overhead hurts more than it helps)

---

In [ ]:
import numpy as np

# Simulate speculative decoding acceptance and speedup
# We model the probability that draft tokens are accepted
# and compute the expected tokens generated per target model call

def simulate_speculative_decoding(
    acceptance_rate: float,   # probability each draft token is accepted
    k: int,                   # number of draft tokens per speculation round
    num_rounds: int = 1000    # simulation rounds
) -> dict:
    """
    Simulate speculative decoding.
    Returns average tokens accepted per target model call.
    """
    tokens_per_call = []

    for _ in range(num_rounds):
        accepted = 0
        for i in range(k):
            if np.random.random() < acceptance_rate:
                accepted += 1
            else:
                break  # first rejection stops the run
        # Always get at least 1 token (resampled from target at rejection point)
        tokens_per_call.append(accepted + 1)

    avg_tokens = np.mean(tokens_per_call)
    # Speedup vs standard decoding (1 token per target call)
    # We assume draft model cost is negligible vs target
    speedup = avg_tokens / 1.0
    return {"avg_tokens": avg_tokens, "speedup": speedup}


print("Speculative Decoding Simulation (K=4 draft tokens)")
print(f"{'Task Type':<25} {'Acceptance Rate':>16} {'Avg Tokens/Call':>16} {'Speedup':>10}")
print("-" * 70)

scenarios = [
    ("Code completion",       0.85),
    ("Translation",           0.75),
    ("Summarisation",         0.65),
    ("Factual Q&A",           0.55),
    ("Creative writing",      0.35),
    ("High-entropy outputs",  0.20),
]

for task, rate in scenarios:
    result = simulate_speculative_decoding(rate, k=4)
    print(f"{task:<25} {rate:>15.0%} {result['avg_tokens']:>15.2f} "
          f"{result['speedup']:>9.2f}x")

In [ ]:
# Show effect of draft token count K on speedup
# There's a sweet spot — more draft tokens helps only if acceptance rate is high

print("Speedup vs Draft Token Count K")
print(f"{'K':>4}" + "".join(f"  α={r:.0%}" for r in [0.9, 0.7, 0.5, 0.3]))
print("-" * 40)

for k in [1, 2, 4, 8, 16]:
    row = f"{k:>4}"
    for rate in [0.9, 0.7, 0.5, 0.3]:
        result = simulate_speculative_decoding(rate, k=k)
        row += f"  {result['speedup']:>5.2f}x"
    print(row)

print()
print("At high acceptance rates (0.9), more draft tokens → more speedup.")
print("At low acceptance rates (0.3), large K wastes draft compute on rejected tokens.")
print("K=4 is a common default — good balance across task types.")

## ✅ What You Just Learned — Section 16

**For AI Engineer interviews:**
- Speculative decoding attacks the sequential bottleneck of autoregressive generation
- The guarantee that output distribution is unchanged is critical — it's not an approximation, it's exact
- The speedup depends on acceptance rate, which depends on task entropy — code and translation benefit most
- It's most effective for latency-sensitive single requests, less so under heavy batching
- Knowing Medusa and prompt lookup decoding as variants shows depth

---

# Section 17: Prompt Caching

## Theory

### What It Is

Prompt caching is a serving-level optimisation that stores the KV cache of a repeated prompt prefix so it doesn't need to be recomputed on subsequent requests. It's the KV cache concept extended across requests rather than within a single generation.

Without prompt caching, every API request recomputes the full KV cache from scratch — even if 99% of the prompt is identical to the previous request (e.g. a long system prompt shared across all users).

With prompt caching, the server detects that the prefix of your prompt matches a cached KV state and skips the prefill computation for those tokens. You only pay for the new tokens that differ from the cached prefix.

### When It Matters

Prompt caching is most valuable when:

- **Long system prompts**: a detailed system prompt of 2,000 tokens sent with every request costs 2,000 tokens of prefill compute per call. With caching, you pay for it once
- **Few-shot examples**: if you include the same 10 examples in every prompt, those can be cached
- **RAG context**: retrieved documents that are reused across multiple follow-up questions
- **Multi-turn conversations**: the growing conversation history can be cached up to the most recent turn

### How Anthropic's Prompt Caching Works

Anthropic exposes prompt caching directly via the API. You mark specific parts of your prompt with a `cache_control` parameter:

```python
{"type": "text", "text": "Your long system prompt...",
 "cache_control": {"type": "ephemeral"}}
```

The first request with this prefix pays full prefill cost and populates the cache. Subsequent requests that share the same prefix get a cache hit and only pay for the tokens after the cache boundary.

Pricing implications (as of 2024):
- Cache write: slightly more expensive than normal input tokens (you pay to store)
- Cache read: significantly cheaper than normal input tokens (~10% of cost)
- Cache TTL: approximately 5 minutes of inactivity before eviction

### The Cache Boundary Rule

Caching only applies to a **prefix** — a contiguous block starting from the beginning of the prompt. You can't cache an arbitrary middle section. This means prompt structure matters:

```
GOOD: [cached system prompt] + [cached few-shot examples] + [new user query]
BAD:  [new user query] + [cached examples]  ← prefix includes new query, no cache hit
```

Structure your prompts so the stable, reusable content comes first.

### Relationship to KV Cache

Prompt caching is essentially a persistent KV cache stored server-side between requests. The mechanics are identical to the within-request KV cache — the server stores the K and V tensors for the cached prefix and reuses them on cache hits. The difference is scope: within-request KV cache lives for the duration of one generation, prompt caching persists across multiple requests.

### Cost Example

Assume a system prompt of 10,000 tokens, sent with 1,000 user requests per day:

```
Without caching: 1,000 × 10,000 = 10,000,000 input tokens/day
With caching:    1 × 10,000 (write) + 999 × 10,000 × 0.1 (reads)
               = 10,000 + 999,000 = ~1,009,000 effective tokens/day
               → ~90% cost reduction on the system prompt portion
```

---

In [ ]:
import anthropic

# Demonstrate prompt caching with the Anthropic API
# We'll use a long system prompt and show cache hit vs miss in the response metadata
# Using claude-haiku-4-5 to keep costs low

client = anthropic.Anthropic()

# A long system prompt that we want to cache
# In production this might be thousands of tokens of documentation or instructions
SYSTEM_PROMPT = """You are an expert AI engineering assistant with deep knowledge of 
LLM inference, model serving, and production ML systems. You help engineers understand 
the tradeoffs between different serving strategies.

Key principles you follow:
1. Always consider the latency vs throughput tradeoff
2. Memory constraints drive most production decisions
3. Quantisation is the first lever for fitting larger models
4. Continuous batching is standard for production serving
5. KV cache management is critical for long-context applications

When answering questions, be specific about memory costs, compute costs, and 
the conditions under which each optimisation is most effective.
""" * 3  # Repeat to make it longer and more worth caching

def ask_with_cache(question: str, use_cache: bool = True) -> dict:
    """Send a question with optional prompt caching and return usage stats."""
    
    system_content = [{"type": "text", "text": SYSTEM_PROMPT}]
    
    if use_cache:
        system_content[0]["cache_control"] = {"type": "ephemeral"}
    
    # Using claude-haiku-4-5 to keep costs low
    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=200,
        system=system_content,
        messages=[{"role": "user", "content": question}]
    )
    
    usage = response.usage
    return {
        "answer": response.content[0].text,
        "input_tokens": usage.input_tokens,
        "cache_creation_tokens": getattr(usage, "cache_creation_input_tokens", 0),
        "cache_read_tokens": getattr(usage, "cache_read_input_tokens", 0),
        "output_tokens": usage.output_tokens,
    }

print(f"System prompt length: ~{len(SYSTEM_PROMPT.split())} words")
print()

# First request — populates the cache
print("Request 1 (cache write):")
result1 = ask_with_cache("What is the main bottleneck in LLM decode phase?", use_cache=True)
print(f"  Answer: {result1['answer'][:100]}...")
print(f"  Input tokens:          {result1['input_tokens']}")
print(f"  Cache creation tokens: {result1['cache_creation_tokens']}")
print(f"  Cache read tokens:     {result1['cache_read_tokens']}")

print()

# Second request — should hit cache for the system prompt
print("Request 2 (cache read — same system prompt, different question):")
result2 = ask_with_cache("When does speculative decoding help most?", use_cache=True)
print(f"  Answer: {result2['answer'][:100]}...")
print(f"  Input tokens:          {result2['input_tokens']}")
print(f"  Cache creation tokens: {result2['cache_creation_tokens']}")
print(f"  Cache read tokens:     {result2['cache_read_tokens']}")
print()
print("On request 2, cache_read_tokens should be non-zero — those tokens")
print("were served from cache at a fraction of normal input token cost.")

What's happening:

We define a long system prompt and send it twice with two different questions. The key addition vs everything we did in Section 8 is this line:

system_content[0]["cache_control"] = {"type": "ephemeral"}

That flag tells Anthropic's servers to cache the KV state for this prefix. On the first call it's a cache write — you pay slightly more than normal to store it. On the second call it's a cache read — the server skips recomputing the prefill for those tokens and you pay roughly 10% of normal input token cost.

The function returns cache_creation_tokens and cache_read_tokens from the usage metadata — those are the two fields that tell you whether you got a cache hit. On request 1 you'd see cache_creation_tokens > 0. On request 2 you'd see cache_read_tokens > 0.

In [5]:
# Model the cost savings from prompt caching
# Using approximate Haiku pricing as of 2024

# Anthropic Haiku pricing (per million tokens, approximate)
PRICE_INPUT_PER_MTK      = 0.25   # $0.25 per million input tokens
PRICE_CACHE_WRITE_PER_MTK = 0.30  # slightly more than input
PRICE_CACHE_READ_PER_MTK  = 0.03  # 88% cheaper than input
PRICE_OUTPUT_PER_MTK      = 1.25  # output tokens always full price

def cost_without_cache(
    system_prompt_tokens: int,
    user_query_tokens: int,
    output_tokens: int,
    num_requests: int
) -> float:
    total_input  = (system_prompt_tokens + user_query_tokens) * num_requests
    total_output = output_tokens * num_requests
    return (total_input * PRICE_INPUT_PER_MTK +
            total_output * PRICE_OUTPUT_PER_MTK) / 1_000_000

def cost_with_cache(
    system_prompt_tokens: int,
    user_query_tokens: int,
    output_tokens: int,
    num_requests: int
) -> float:
    # First request: write to cache
    cache_write_cost = system_prompt_tokens * PRICE_CACHE_WRITE_PER_MTK / 1_000_000
    # Subsequent requests: read from cache for system prompt, normal for query
    cache_read_cost  = system_prompt_tokens * (num_requests - 1) * PRICE_CACHE_READ_PER_MTK / 1_000_000
    query_cost       = user_query_tokens * num_requests * PRICE_INPUT_PER_MTK / 1_000_000
    output_cost      = output_tokens * num_requests * PRICE_OUTPUT_PER_MTK / 1_000_000
    return cache_write_cost + cache_read_cost + query_cost + output_cost

print("Cost savings from prompt caching (Haiku pricing)")
print(f"Assumes: 50 token user queries, 200 token outputs")
print()
print(f"{'System Prompt':>15} {'Requests':>10} {'No Cache':>12} {'With Cache':>12} {'Savings':>10}")
print("-" * 62)

for sys_tokens in [500, 2000, 10000]:
    for reqs in [100, 1000, 10000]:
        no_cache   = cost_without_cache(sys_tokens, 50, 200, reqs)
        with_cache = cost_with_cache(sys_tokens, 50, 200, reqs)
        savings    = (no_cache - with_cache) / no_cache
        print(f"{sys_tokens:>12} tk {reqs:>10,} ${no_cache:>10.4f} "
              f"${with_cache:>10.4f} {savings:>9.1%}")

Cost savings from prompt caching (Haiku pricing)
Assumes: 50 token user queries, 200 token outputs

  System Prompt   Requests     No Cache   With Cache    Savings
--------------------------------------------------------------
         500 tk        100 $    0.0387 $    0.0279     28.0%
         500 tk      1,000 $    0.3875 $    0.2776     28.4%
         500 tk     10,000 $    3.8750 $    2.7751     28.4%
        2000 tk        100 $    0.0762 $    0.0328     57.0%
        2000 tk      1,000 $    0.7625 $    0.3230     57.6%
        2000 tk     10,000 $    7.6250 $    3.2255     57.7%
       10000 tk        100 $    0.2762 $    0.0590     78.7%
       10000 tk      1,000 $    2.7625 $    0.5652     79.5%
       10000 tk     10,000 $   27.6250 $    5.6277     79.6%


What's happening:

We model the pricing difference between cached and uncached requests across different system prompt sizes and request volumes.

cost_without_cache is simple — every request pays full price for the system prompt tokens plus query tokens plus output tokens. The system prompt cost scales linearly with requests.

cost_with_cache splits the system prompt cost into two parts. The first request pays PRICE_CACHE_WRITE_PER_MTK (slightly more than normal) to populate the cache. Every subsequent request pays PRICE_CACHE_READ_PER_MTK (88% cheaper than normal) for those same tokens. 

The user query and output tokens always pay full price regardless — only the cached prefix gets the discount.

The table it prints shows savings across combinations of system prompt length (500, 2000, 10000 tokens) and request volume (100, 1000, 10000 requests). The pattern you'll see: savings are negligible at low volume and short prompts, enormous at high volume and long prompts. At 10,000 tokens and 10,000 requests the saving is around 80-90% on the input side.

## ✅ What You Just Learned — Section 17

**For AI Engineer interviews:**
- Prompt caching is the easiest production win when you have repeated prompt prefixes — no code changes beyond adding a `cache_control` flag
- It's KV cache extended across requests — same mechanism, different scope
- The prefix constraint means prompt structure matters: stable content first, dynamic content last
- At scale (10k+ requests/day with long system prompts), prompt caching can reduce costs by 80-90% on the input side
- Cache TTL means it's not suitable for low-traffic applications — the cache evicts before it pays off

---

## Summary: Production Inference Optimisation Decision Framework

| Problem | Solution | Primary Benefit |
|---|---|---|
| Slow token generation | KV Cache | Eliminates redundant K,V recomputation |
| Model doesn't fit in VRAM | Quantisation (INT8/INT4) | 2-4× memory reduction |
| Low GPU utilisation | Continuous batching | Near-100% GPU utilisation |
| Memory fragmentation | PagedAttention | Higher effective batch size |
| High latency on single requests | Speculative decoding | 2-3× faster for predictable outputs |
| High cost with repeated prompts | Prompt caching | 80-90% reduction on cached tokens |

In practice these are layered — a production serving stack uses all of them simultaneously. vLLM gives you continuous batching + PagedAttention out of the box. Quantisation is applied at model load time. Speculative decoding is optional. Prompt caching is an API-level feature.

The interview question is usually about which lever to reach for first given a specific constraint: cost → quantisation + prompt caching; latency → speculative decoding + KV cache; throughput → continuous batching + PagedAttention.